In [34]:
# If .secrets/fmp.py is a local file, add it to sys.path first:
import sys
from pathlib import Path
    
import argparse
import os
import json
from datetime import datetime
import requests

try:
    import yfinance as yf
except ImportError:
    yf = None

root_dir = Path.cwd().parent 

# Create raw_data directory if it doesn't exist
raw_data_dir = root_dir / "raw_data"
raw_data_dir.mkdir(exist_ok=True)

In [ ]:
# secrets = os.getenv(secrets_dir)
def load_secret(path, key_name, storage_key):
    '''
        Load a secret from a JSON file and store it in an environment variable.
        Args:
            path (str): The path to the JSON file containing the secret.  
            key_name (str): The key in the JSON file that contains the secret value.
            storage_key (str): The name of the environment variable to store the secret value in.
    '''
    try:
        with open(path, "r", encoding="utf-8") as f:
            secret = json.load(f)
            os.environ.update({storage_key: secret[key_name]})
    except Exception as e:
        print(f"Error loading secret from {path}: {e}")

# Get the FMP API key from the .secrets/fmp.json file and store it in the environment variable FMP_API_KEY
load_secret(os.path.join(root_dir,'.secrets', "fmp.json"), "fmp_secret", "FMP_API_KEY")

In [31]:
def _filter_by_years(df, years):
    if years is None or len(years) == 0:
        return df
    keep = []
    for col in df.columns:
        if isinstance(col, datetime):
            year = col.year
        else:
            try:
                year = int(str(col)[:4])
            except Exception:
                continue
        if year in years:
            keep.append(col)
    return df.loc[:, keep]

def get_yfinance_financials(ticker, years=None):
    if yf is None:
        raise RuntimeError("yfinance is not installed. pip install yfinance")

    ty = yf.Ticker(ticker)
    results = {
        "ticker": ticker,
        "source": "yfinance",
        "company_info": ty.info if hasattr(ty, "info") else {},
    }

    statements = {
        "balance_sheet": getattr(ty, "balance_sheet", None),
        "income_statement": getattr(ty, "financials", None),
        "cash_flow_statement": getattr(ty, "cashflow", None),
        "earnings": getattr(ty, "earnings", None),
        "quarterly_balance_sheet": getattr(ty, "quarterly_balance_sheet", None),
        "quarterly_financials": getattr(ty, "quarterly_financials", None),
        "quarterly_cashflow": getattr(ty, "quarterly_cashflow", None),
    }

    for label, df in statements.items():
        if df is None:
            results[label] = None
            continue
        filtered = _filter_by_years(df, years) if years else df
        results[label] = json.loads(filtered.fillna("").to_json(orient="index", date_format="iso"))

    return results


def get_fmp_financials(ticker, years=None):
    results = {
        "ticker": ticker,
        "source": "fmp",
    }

    items = {
        "balance_sheet": "balance-sheet-statement",
        "income_statement": "income-statement",
        "cash_flow_statement": "cash-flow-statement",
        "ratios": "ratios",
        "cash_flow_ratios": "cash-flow-statement",
        "profile": "profile",
    }

    for label, endpoint in items.items():
        try:
            data = _fmp_request(endpoint, ticker)
            if years:
                data = [d for d in data if d.get("calendarYear") in years or d.get("date") and int(d.get("date")[:4]) in years]
            results[label] = data
        except Exception as ex:
            results[label] = {"error": str(ex)}

    return results


def get_financials(ticker, years=None, source="yfinance"):
    years_set = {int(y) for y in years} if years else None
    if source == "yfinance":
        return get_yfinance_financials(ticker, years_set)
    elif source == "fmp":
        return get_fmp_financials(ticker, years_set)
    else:
        raise ValueError("Unknown source: choose 'yfinance' or 'fmp'")



def _fmp_request(path, ticker, limit=1000):
    key = os.environ.get("FMP_API_KEY", "")
    if not key:
        raise RuntimeError("Environment variable FMP_API_KEY is required for source=fmp")
    url = f"https://financialmodelingprep.com/api/v3/{path}/{ticker}?limit={limit}&apikey={key}"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.json()



def run(tickers, years=None, source="yfinance", output=None, pretty=False):
    if isinstance(tickers, str):
        tickers = [tickers]

    all_data = []
    for t in tickers:
        print(f"Fetching {t} from {source}...", flush=True)
        data = get_financials(t, years=years, source=source)
        all_data.append(data)

    if output:
        with open(output, "w", encoding="utf-8") as f:
            if pretty:
                json.dump(all_data, f, indent=2)
            else:
                json.dump(all_data, f)
        print(f"Saved results to: {output}")

    return all_data


def parse_args():
    p = argparse.ArgumentParser(description="Download US-listed company financial statements.")
    p.add_argument("--tickers", nargs="+", required=True, help="Ticker symbols, e.g., AAPL MSFT")
    p.add_argument("--years", nargs="*", type=int, default=[], help="Year filters, e.g., 2020 2021")
    p.add_argument("--source", choices=["yfinance", "fmp"], default="yfinance", help="Data source")
    p.add_argument("--output", default="financials.json", help="Output JSON file")
    p.add_argument("--pretty", action="store_true", help="Pretty-print JSON")
    return p.parse_args()


In [32]:

# if __name__ == "__main__":
# args = parse_args()
args = {
    "tickers": ["AAPL", "MSFT"],
    "years": [2022, 2023, 2024, 2025, 2026],
    "source": "yfinance",
    "output": f"{root_dir}/raw_data/financials.json",
    "pretty": False,
}
# args['tickers'] = ["AAPL", "MSFT"]
# args.years = [2020, 2021]
# args.source = "yfinance"
# args.output = "financials.json"
# args.pretty = False

# For when running as a script, uncomment the following lines:
# result = run(args.tickers, years=args.years, source=args.source, output=args.output, pretty=args.pretty)

result = run(args['tickers'], years=args['years'], source=args['source'], output=args['output'], pretty=args['pretty'])
print(json.dumps(result, indent=2) if args['pretty'] else json.dumps(result))


Fetching AAPL from yfinance...


d:\Documents\Projects\finance-data-pipeline\.venv\Lib\site-packages\yfinance\scrapers\fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


Fetching MSFT from yfinance...


d:\Documents\Projects\finance-data-pipeline\.venv\Lib\site-packages\yfinance\scrapers\fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


Saved results to: d:\Documents\Projects\finance-data-pipeline/raw_data/financials.json
[{"ticker": "AAPL", "source": "yfinance", "company_info": {"address1": "One Apple Park Way", "city": "Cupertino", "state": "CA", "zip": "95014", "country": "United States", "phone": "(408) 996-1010", "website": "https://www.apple.com", "industry": "Consumer Electronics", "industryKey": "consumer-electronics", "industryDisp": "Consumer Electronics", "sector": "Technology", "sectorKey": "technology", "sectorDisp": "Technology", "longBusinessSummary": "Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare s

# Transforming data

In [ ]:
balance_sheet_dtype = {
    "Treasury Shares Number": "Int64",
    "Ordinary Shares Number": "Int64",
    "Share Issued": "Int64",
    "Net Debt": "float64",
    "Total Debt": "float64",
    "Tangible Book Value": "float64",
    "Invested Capital": "float64",
    "Working Capital": "float64",
    "Net Tangible Assets": "float64",
    "Capital Lease Obligations": "float64",
    "Common Stock Equity": "float64",
    "Total Capitalization": "float64",
    "Total Equity Gross Minority Interest": "float64",
    "Stockholders Equity": "float64",
    "Gains Losses Not Affecting Retained Earnings": "float64",
    "Other Equity Adjustments": "float64",
    "Retained Earnings": "float64",
    "Capital Stock": "float64",
    "Common Stock": "float64",
    "Total Liabilities Net Minority Interest": "float64",
    "Total Non Current Liabilities Net Minority Interest": "float64",
    "Other Non Current Liabilities": "float64",
    "Tradeand Other Payables Non Current": "float64",
    "Long Term Debt And Capital Lease Obligation": "float64",
    "Long Term Capital Lease Obligation": "float64",
    "Long Term Debt": "float64",
    "Current Liabilities": "float64",
    "Other Current Liabilities": "float64",
    "Current Deferred Liabilities": "float64",
    "Current Deferred Revenue": "float64",
    "Current Debt And Capital Lease Obligation": "float64",
    "Current Capital Lease Obligation": "float64",
    "Current Debt": "float64",
    "Other Current Borrowings": "float64",
    "Commercial Paper": "float64",
    "Payables And Accrued Expenses": "float64",
    "Current Accrued Expenses": "float64",
    "Payables": "float64",
    "Total Tax Payable": "float64",
    "Income Tax Payable": "float64",
    "Accounts Payable": "float64",
    "Total Assets": "float64",
    "Total Non Current Assets": "float64",
    "Other Non Current Assets": "float64",
    "Non Current Deferred Assets": "float64",
    "Non Current Deferred Taxes Assets": "float64",
    "Investments And Advances": "float64",
    "Other Investments": "float64",
    "Investmentin Financial Assets": "float64",
    "Available For Sale Securities": "float64",
    "Net PPE": "float64",
    "Accumulated Depreciation": "float64",
    "Gross PPE": "float64",
    "Leases": "float64",
    "Other Properties": "float64",
    "Machinery Furniture Equipment": "float64",
    "Land And Improvements": "float64",
    "Properties": "float64",
    "Current Assets": "float64",
    "Other Current Assets": "float64",
    "Inventory": "float64",
    "Receivables": "float64",
    "Other Receivables": "float64",
    "Accounts Receivable": "float64",
    "Cash Cash Equivalents And Short Term Investments": "float64",
    "Other Short Term Investments": "float64",
    "Cash And Cash Equivalents": "float64",
    "Cash Equivalents": "float64",
    "Cash Financial": "float64"
}

In [17]:
import pandas as pd

balance_sheet_dtype = {
    "Treasury Shares Number": "Int64",
    "Ordinary Shares Number": "Int64",
    "Share Issued": "Int64",
    "Net Debt": "float64",
    "Total Debt": "float64",
    "Tangible Book Value": "float64",
    "Invested Capital": "float64",
    "Working Capital": "float64",
    "Net Tangible Assets": "float64",
    "Capital Lease Obligations": "float64",
    "Common Stock Equity": "float64",
    "Total Capitalization": "float64",
    "Total Equity Gross Minority Interest": "float64",
    "Stockholders Equity": "float64",
    "Gains Losses Not Affecting Retained Earnings": "float64",
    "Other Equity Adjustments": "float64",
    "Retained Earnings": "float64",
    "Capital Stock": "float64",
    "Common Stock": "float64",
    "Total Liabilities Net Minority Interest": "float64",
    "Total Non Current Liabilities Net Minority Interest": "float64",
    "Other Non Current Liabilities": "float64",
    "Tradeand Other Payables Non Current": "float64",
    "Long Term Debt And Capital Lease Obligation": "float64",
    "Long Term Capital Lease Obligation": "float64",
    "Long Term Debt": "float64",
    "Current Liabilities": "float64",
    "Other Current Liabilities": "float64",
    "Current Deferred Liabilities": "float64",
    "Current Deferred Revenue": "float64",
    "Current Debt And Capital Lease Obligation": "float64",
    "Current Capital Lease Obligation": "float64",
    "Current Debt": "float64",
    "Other Current Borrowings": "float64",
    "Commercial Paper": "float64",
    "Payables And Accrued Expenses": "float64",
    "Current Accrued Expenses": "float64",
    "Payables": "float64",
    "Total Tax Payable": "float64",
    "Income Tax Payable": "float64",
    "Accounts Payable": "float64",
    "Total Assets": "float64",
    "Total Non Current Assets": "float64",
    "Other Non Current Assets": "float64",
    "Non Current Deferred Assets": "float64",
    "Non Current Deferred Taxes Assets": "float64",
    "Investments And Advances": "float64",
    "Other Investments": "float64",
    "Investmentin Financial Assets": "float64",
    "Available For Sale Securities": "float64",
    "Net PPE": "float64",
    "Accumulated Depreciation": "float64",
    "Gross PPE": "float64",
    "Leases": "float64",
    "Other Properties": "float64",
    "Machinery Furniture Equipment": "float64",
    "Land And Improvements": "float64",
    "Properties": "float64",
    "Current Assets": "float64",
    "Other Current Assets": "float64",
    "Inventory": "float64",
    "Receivables": "float64",
    "Other Receivables": "float64",
    "Accounts Receivable": "float64",
    "Cash Cash Equivalents And Short Term Investments": "float64",
    "Other Short Term Investments": "float64",
    "Cash And Cash Equivalents": "float64",
    "Cash Equivalents": "float64",
    "Cash Financial": "float64"
}

income_statement_dtype = {
    "Total Revenue": "float64",
    "Operating Revenue": "float64",
    "Revenue": "float64",
    "Cost Of Revenue": "float64",
    "Gross Profit": "float64",
    "Selling General And Administration": "float64",
    "Research And Development": "float64",
    "Operating Expenses": "float64",
    "Operating Income": "float64",
    "Total Operating Expenses": "float64",
    "Operating Income Or Loss": "float64",
    "Pretax Income": "float64",
    "Income Tax Expense": "float64",
    "Income From Continuing Operations": "float64",
    "Net Income From Continuing Operations": "float64",
    "Discontinued Operation": "float64",
    "Extraordinary Items": "float64",
    "Effect Of Accounting Charges": "float64",
    "Other Items": "float64",
    "Net Income": "float64",
    "Preferred Stock And Other Adjustments": "float64",
    "Net Income Common Stockholders": "float64",
    "Basic EPS": "float64",
    "Diluted EPS": "float64",
    "Basic Average Shares": "Int64",
    "Diluted Average Shares": "Int64",
    "Total Costs": "float64",
    "Total Expenses": "float64",
    "Minority Interest Expense": "float64",
    "Tax Provision": "float64",
    "Other Income Expense": "float64",
    "Interest Expense": "float64",
    "Interest Income": "float64",
}

cash_flow_statement_dtype = {
    "Operating Cash Flow": "float64",
    "Cash Flow From Continuing Operating Activities": "float64",
    "Net Income": "float64",
    "Depreciation": "float64",
    "Deferred Income Taxes": "float64",
    "Stock Based Compensation": "float64",
    "Other Non Cash Items": "float64",
    "Changes In Working Capital": "float64",
    "Accounts Receivable": "float64",
    "Inventory": "float64",
    "Accounts Payable": "float64",
    "Other Working Capital": "float64",
    "Other Operating Activities": "float64",
    "Investing Cash Flow": "float64",
    "Investing Activities Other": "float64",
    "Capital Expenditures": "float64",
    "Net Acquisitions": "float64",
    "Other Investments": "float64",
    "Cash Flow From Continuing Investing Activities": "float64",
    "Financing Cash Flow": "float64",
    "Cash Dividends Paid": "float64",
    "Cash Paid For Taxes": "float64",
    "Issuance Of Stock": "float64",
    "Repurchase Of Stock": "float64",
    "Issuance Of Debt": "float64",
    "Repayment Of Debt": "float64",
    "Other Financing Activities": "float64",
    "Cash Flow From Continuing Financing Activities": "float64",
    "Effect Of Exchange Rate On Cash": "float64",
    "Net Change In Cash": "float64",
    "Free Cash Flow": "float64",
    "Common Stock Issued": "Int64",
    "Common Stock Repurchased": "Int64",
}

profile_dtype = {
    "Full Time Employees": "Int64",
    "Full-time Employees": "Int64",
    "Market Cap": "float64",
    "Market Capitalization": "float64",
    "Revenue Per Share": "float64",
    "Profit Margin": "float64",
    "Payout Ratio": "float64",
    "Return On Equity": "float64",
    "Return On Assets": "float64",
    "Debt To Equity": "float64",
    "Current Ratio": "float64",
    "Quick Ratio": "float64",
    "Book Value": "float64",
    "Audit Risk": "Int64",
    "Board Risk": "Int64",
    "Compensation Risk": "Int64",
    "Shareholder Rights Risk": "Int64",
    "Overall Risk": "Int64",
}

try:
    with open("financials.json", "r") as f:
        data = json.load(f)
        
        balance_sheet = pd.DataFrame(data[0].get('balance_sheet', {}))
        income_statement = pd.DataFrame(data[0].get('income_statement', {}))
        cash_flow_statement = pd.DataFrame(data[0].get('cash_flow_statement', {}))
        ratios = pd.DataFrame(data[0].get('ratios', {}))
        cash_flow_ratios = pd.DataFrame(data[0].get('cash_flow_ratios', {}))
        profile = pd.DataFrame(data[0].get('profile', {}))

        # Apply dtypes to balance_sheet
        for col, typ in balance_sheet_dtype.items():
            if col in balance_sheet.columns:
                if typ == "float64":
                    balance_sheet[col] = pd.to_numeric(balance_sheet[col], errors='coerce')
                elif typ == "Int64":
                    balance_sheet[col] = pd.to_numeric(balance_sheet[col], errors='coerce').astype('Int64')

        # Apply dtypes to income_statement
        for col, typ in income_statement_dtype.items():
            if col in income_statement.columns:
                if typ == "float64":
                    income_statement[col] = pd.to_numeric(income_statement[col], errors='coerce')
                elif typ == "Int64":
                    income_statement[col] = pd.to_numeric(income_statement[col], errors='coerce').astype('Int64')

        # Apply dtypes to cash_flow_statement
        for col, typ in cash_flow_statement_dtype.items():
            if col in cash_flow_statement.columns:
                if typ == "float64":
                    cash_flow_statement[col] = pd.to_numeric(cash_flow_statement[col], errors='coerce')
                elif typ == "Int64":
                    cash_flow_statement[col] = pd.to_numeric(cash_flow_statement[col], errors='coerce').astype('Int64')

        # Apply dtypes to profile
        for col, typ in profile_dtype.items():
            if col in profile.columns:
                if typ == "float64":
                    profile[col] = pd.to_numeric(profile[col], errors='coerce')
                elif typ == "Int64":
                    profile[col] = pd.to_numeric(profile[col], errors='coerce').astype('Int64')

        # Reset index to create report_date column
        balance_sheet = balance_sheet.reset_index().rename(columns={'index': 'report_date'})
        balance_sheet['report_date'] = pd.to_datetime(balance_sheet['report_date'])
        
        income_statement = income_statement.reset_index().rename(columns={'index': 'report_date'})
        income_statement['report_date'] = pd.to_datetime(income_statement['report_date'])
        
        cash_flow_statement = cash_flow_statement.reset_index().rename(columns={'index': 'report_date'})
        cash_flow_statement['report_date'] = pd.to_datetime(cash_flow_statement['report_date'])
        
        if not profile.empty:
            profile = profile.reset_index().rename(columns={'index': 'report_date'})
            profile['report_date'] = pd.to_datetime(profile['report_date'])

        # print(balance_sheet)

except Exception as ex:
    print(f"Error loading financials.json: {ex}")
    


balance_sheet.head()

,report_date,Treasury Shares Number,Ordinary Shares Number,Share Issued,Net Debt,Total Debt,Tangible Book Value,Invested Capital,Working Capital,Net Tangible Assets,...,Other Current Assets,Inventory,Receivables,Other Receivables,Accounts Receivable,Cash Cash Equivalents And Short Term Investments,Other Short Term Investments,Cash And Cash Equivalents,Cash Equivalents,Cash Financial
0,2025-09-30,<NA>,14773260000,14773260000,6.272300e+10,9.865700e+10,7.373300e+10,1.723900e+11,-1.767400e+10,7.373300e+10,...,1.458500e+10,5.718000e+09,7.295700e+10,3.318000e+10,3.977700e+10,5.469700e+10,1.876300e+10,3.593400e+10,7.667000e+09,2.826700e+10
1,2024-09-30,<NA>,15116786000,15116786000,7.668600e+10,1.066290e+11,5.695000e+10,1.635790e+11,-2.340500e+10,5.695000e+10,...,1.428700e+10,7.286000e+09,6.624300e+10,3.283300e+10,3.341000e+10,6.517100e+10,3.522800e+10,2.994300e+10,2.744000e+09,2.719900e+10
2,2023-09-30,0,15550061000,15550061000,8.112300e+10,1.110880e+11,6.214600e+10,1.732340e+11,-1.742000e+09,6.214600e+10,...,1.469500e+10,6.331000e+09,6.098500e+10,3.147700e+10,2.950800e+10,6.155500e+10,3.159000e+10,2.996500e+10,1.606000e+09,2.835900e+10
3,2022-09-30,<NA>,15943425000,15943425000,9.642300e+10,1.324800e+11,5.067200e+10,1.707410e+11,-1.857700e+10,5.067200e+10,...,2.122300e+10,4.946000e+09,6.093200e+10,3.274800e+10,2.818400e+10,4.830400e+10,2.465800e+10,2.364600e+10,5.100000e+09,1.854600e+10
4,2021-09-30,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
income_statement.head(15)


,report_date,Tax Effect Of Unusual Items,Tax Rate For Calcs,Normalized EBITDA,Net Income From Continuing Operation Net Minority Interest,Reconciled Depreciation,Reconciled Cost Of Revenue,EBITDA,EBIT,Net Interest Income,...,Interest Expense Non Operating,Interest Income Non Operating,Operating Income,Operating Expense,Research And Development,Selling General And Administration,Gross Profit,Cost Of Revenue,Total Revenue,Operating Revenue
0,2025-09-30,0.0,0.156,144748000000.0,112010000000.0,11698000000.0,220960000000.0,144748000000.0,133050000000.0,,...,,,1.330500e+11,62151000000.0,3.455000e+10,2.760100e+10,1.952010e+11,2.209600e+11,4.161610e+11,4.161610e+11
1,2024-09-30,0.0,0.241,134661000000.0,93736000000.0,11445000000.0,210352000000.0,134661000000.0,123216000000.0,,...,,,1.232160e+11,57467000000.0,3.137000e+10,2.609700e+10,1.806830e+11,2.103520e+11,3.910350e+11,3.910350e+11
2,2023-09-30,0.0,0.147,125820000000.0,96995000000.0,11519000000.0,214137000000.0,125820000000.0,114301000000.0,-183000000.0,...,3933000000.0,3750000000.0,1.143010e+11,54847000000.0,2.991500e+10,2.493200e+10,1.691480e+11,2.141370e+11,3.832850e+11,3.832850e+11
3,2022-09-30,0.0,0.162,130541000000.0,99803000000.0,11104000000.0,223546000000.0,130541000000.0,119437000000.0,-106000000.0,...,2931000000.0,2825000000.0,1.194370e+11,51345000000.0,2.625100e+10,2.509400e+10,1.707820e+11,2.235460e+11,3.943280e+11,3.943280e+11
4,2021-09-30,,,,,,,,,198000000.0,...,2645000000.0,2843000000.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
cash_flow_statement.head()

,report_date,Free Cash Flow,Repurchase Of Capital Stock,Repayment Of Debt,Issuance Of Debt,Issuance Of Capital Stock,Capital Expenditure,Interest Paid Supplemental Data,Income Tax Paid Supplemental Data,End Cash Position,...,Change In Inventory,Change In Receivables,Changes In Account Receivables,Other Non Cash Items,Stock Based Compensation,Deferred Tax,Deferred Income Tax,Depreciation Amortization Depletion,Depreciation And Amortization,Net Income From Continuing Operations
0,2025-09-30,9.876700e+10,-90711000000.0,-1.093200e+10,4.481000e+09,,-12715000000.0,,43369000000.0,35934000000.0,...,1400000000.0,-7029000000.0,-6682000000.0,-8.900000e+07,1.286300e+10,,,11698000000.0,11698000000.0,112010000000.0
1,2024-09-30,1.088070e+11,-94949000000.0,-9.958000e+09,0.000000e+00,,-9447000000.0,,26102000000.0,29943000000.0,...,-1046000000.0,-5144000000.0,-3788000000.0,-2.266000e+09,1.168800e+10,,,11445000000.0,11445000000.0,93736000000.0
2,2023-09-30,9.958400e+10,-77550000000.0,-1.115100e+10,5.228000e+09,,-10959000000.0,3803000000.0,18679000000.0,30737000000.0,...,-1618000000.0,-417000000.0,-1688000000.0,-2.227000e+09,1.083300e+10,,,11519000000.0,11519000000.0,96995000000.0
3,2022-09-30,1.114430e+11,-89402000000.0,-9.543000e+09,5.465000e+09,,-10708000000.0,2865000000.0,19573000000.0,24977000000.0,...,1484000000.0,-9343000000.0,-1823000000.0,1.006000e+09,9.038000e+09,895000000.0,895000000.0,11104000000.0,11104000000.0,99803000000.0
4,2021-09-30,NaN,,NaN,NaN,1105000000.0,,2687000000.0,,,...,,,,NaN,NaN,-4774000000.0,-4774000000.0,,,


In [22]:
profile

""


In [ ]:
with open("output.txt", "w") as f:
    for c in balance_sheet.columns:
        print(f"\"{c}\": \"{type(c)}\"", file=f)

balance_sheet.head(20).to_csv("balance_sheet_output.csv", index=True)